# Notebook 03: Vision Transformer (ViT-B/16) Fine-Tuning
**Project:** Deep Learning-Based COVID-19 Detection from Chest X-ray Images Using Explainable AI  
**Student:** Channabasavanna Santosh Pawate (16150425)  

## Objective
Fine-tune `google/vit-base-patch16-224-in21k` on the COVID-19 chest X-ray dataset.

**Strategy:**
- Stage 1: Freeze ViT body, train only classifier head (5 epochs, lr=1e-4)
- Stage 2: Unfreeze all layers, fine-tune end-to-end (10 epochs, lr=2e-5)

**Loss:** Class-weighted cross-entropy to handle class imbalance

In [ ]:
import os
import sys
import time
import json
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import ViTForImageClassification, ViTConfig

# Add model directory to path
sys.path.append('..')

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'\nPyTorch: {torch.__version__}')
print(f'Seed: {SEED}')

## 1. Load Data

In [ ]:
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])


class ChestXrayDataset(Dataset):
    CLASSES = ['Normal', 'COVID-19', 'Viral Pneumonia']
    CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}

    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        for cls in self.CLASSES:
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.exists(cls_dir):
                continue
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(cls_dir, fname),
                                         self.CLASS2IDX[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

    def get_class_weights(self):
        from collections import Counter
        label_counts = Counter(label for _, label in self.samples)
        total = len(self.samples)
        weights = [total / (len(self.CLASSES) * label_counts.get(i, 1))
                   for i in range(len(self.CLASSES))]
        return torch.FloatTensor(weights)


DATA_ROOT = '../data'
BATCH_SIZE = 32

if os.path.exists(DATA_ROOT):
    train_ds = ChestXrayDataset(os.path.join(DATA_ROOT, 'train'), train_transform)
    val_ds   = ChestXrayDataset(os.path.join(DATA_ROOT, 'val'),   val_transform)
    test_ds  = ChestXrayDataset(os.path.join(DATA_ROOT, 'test'),  val_transform)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    
    class_weights = train_ds.get_class_weights().to(DEVICE)
    print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
    print(f'Class weights: {class_weights.tolist()}')
else:
    print('Dataset not found. Please download and prepare the data first.')
    print('See Notebook 01 for dataset sources.')

## 2. Load Pretrained ViT-B/16

In [ ]:
NUM_CLASSES = 3
ID2LABEL = {0: 'Normal', 1: 'COVID-19', 2: 'Viral Pneumonia'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

print('Loading google/vit-base-patch16-224-in21k ...')
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224-in21k',
    num_labels=NUM_CLASSES,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'\nModel architecture: ViT-B/16')
print(f'  Patch size: 16×16')
print(f'  Input size: 224×224')
print(f'  Patches: {(224//16)**2} = 196')
print(f'  Hidden dim: 768')
print(f'  Attention heads: 12')
print(f'  Transformer blocks: 12')
print(f'  Output classes: {NUM_CLASSES}')

## 3. Training Utilities

In [ ]:
def freeze_encoder(model):
    """Freeze all ViT layers except the classifier head."""
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Stage 1: Frozen encoder. Trainable params: {trainable:,}')


def unfreeze_all(model):
    """Unfreeze all model layers for full fine-tuning."""
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Stage 2: All layers unfrozen. Trainable params: {trainable:,}')


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(pixel_values=images)
        logits = outputs.logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = logits.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(pixel_values=images)
        logits = outputs.logits
        loss = criterion(logits, labels)
        running_loss += loss.item() * images.size(0)
        _, preds = logits.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


print('Training utilities defined.')

## 4. Stage 1 Training — Classifier Head Only

In [ ]:
if not os.path.exists(DATA_ROOT):
    print('Dataset not available. Training skipped.')
else:
    # Stage 1 config
    STAGE1_EPOCHS = 5
    STAGE1_LR = 1e-4
    
    freeze_encoder(model)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=STAGE1_LR, weight_decay=0.01
    )
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    
    print(f'\n=== Stage 1: Head-only training ({STAGE1_EPOCHS} epochs, lr={STAGE1_LR}) ===')
    for epoch in range(1, STAGE1_EPOCHS + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
        elapsed = time.time() - t0
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), '../model/saved/vit_covid_stage1_best.pth')
        
        print(f'Epoch {epoch}/{STAGE1_EPOCHS} | '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | '
              f'{elapsed:.1f}s')
    
    print(f'\nBest Stage 1 Val Acc: {best_val_acc:.4f}')

## 5. Stage 2 Training — Full Fine-Tuning

In [ ]:
if not os.path.exists(DATA_ROOT):
    print('Dataset not available. Training skipped.')
else:
    # Stage 2 config
    STAGE2_EPOCHS = 10
    STAGE2_LR = 2e-5
    
    unfreeze_all(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=STAGE2_LR, weight_decay=0.01)
    
    # Cosine annealing scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STAGE2_EPOCHS)
    
    print(f'\n=== Stage 2: Full fine-tuning ({STAGE2_EPOCHS} epochs, lr={STAGE2_LR}) ===')
    for epoch in range(1, STAGE2_EPOCHS + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()
        elapsed = time.time() - t0
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), '../model/saved/vit_covid_best.pth')
        
        lr_now = scheduler.get_last_lr()[0]
        print(f'Epoch {epoch}/{STAGE2_EPOCHS} | '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | '
              f'LR: {lr_now:.2e} | {elapsed:.1f}s')
    
    # Save final model
    model.save_pretrained('../model/saved/vit_covid_final')
    with open('../model/saved/training_history.json', 'w') as f:
        json.dump(history, f)
    
    print(f'\nBest Overall Val Acc: {best_val_acc:.4f}')
    print('Model saved to ../model/saved/vit_covid_final/')
    print('Training history saved to ../model/saved/training_history.json')

## 6. Training Curves

In [ ]:
def plot_training_curves(history, stage1_epochs=5):
    """Plot loss and accuracy curves for both training stages."""
    total_epochs = len(history['train_loss'])
    epochs = list(range(1, total_epochs + 1))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss
    axes[0].plot(epochs, history['train_loss'], 'b-o', markersize=4, label='Train Loss')
    axes[0].plot(epochs, history['val_loss'],   'r-s', markersize=4, label='Val Loss')
    axes[0].axvline(x=stage1_epochs + 0.5, color='gray', linestyle='--', alpha=0.7, label='Stage boundary')
    axes[0].fill_betweenx([0, max(history['train_loss']) * 1.1],
                           0.5, stage1_epochs + 0.5, alpha=0.05, color='blue', label='Stage 1')
    axes[0].fill_betweenx([0, max(history['train_loss']) * 1.1],
                           stage1_epochs + 0.5, total_epochs + 0.5, alpha=0.05, color='orange', label='Stage 2')
    axes[0].set_title('Training & Validation Loss', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(epochs, [a * 100 for a in history['train_acc']], 'b-o', markersize=4, label='Train Acc')
    axes[1].plot(epochs, [a * 100 for a in history['val_acc']],   'r-s', markersize=4, label='Val Acc')
    axes[1].axvline(x=stage1_epochs + 0.5, color='gray', linestyle='--', alpha=0.7)
    axes[1].axhline(y=95, color='green', linestyle=':', alpha=0.7, label='95% target')
    axes[1].set_title('Training & Validation Accuracy', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_ylim([0, 105])
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle('ViT-B/16 Fine-Tuning Progress (Stage 1 + Stage 2)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/figures/training_curves.png', bbox_inches='tight', dpi=150)
    plt.show()


# Load history if it was saved
history_path = '../model/saved/training_history.json'
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    plot_training_curves(history)
    print(f'Final train acc: {history["train_acc"][-1]*100:.2f}%')
    print(f'Final val acc:   {history["val_acc"][-1]*100:.2f}%')
else:
    print('Training history not found. Run training cells above first.')